## Muon_skript_reco_test

In [1]:
import sqlite3
import pandas as pd

db_path = "/lustre/hpc/project/icecube/MonteCarlo2022/databases/Muon/Muon_merged.db"
#db_path = "/groups/icecube/janikh/PREP/I3_read_out/Data_storage/merged/events.db"
# mode=ro: read-only
# immutable=1: SQLite versucht keine Locks/Journale anzulegen (sehr hilfreich auf HPC/readonly)
uri = f"file:{db_path}?mode=ro&immutable=1"

conn = sqlite3.connect(uri, uri=True)

# optional: extra Safety – verhindert Writes auch innerhalb der Session
conn.execute("PRAGMA query_only = ON;")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

tables.head()

,name
0,SplitInIcePulses
1,truth


In [2]:
pd.read_sql("PRAGMA table_info(SplitInIcePulses);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,charge,,1,None,0
1,1,dom_time,,1,None,0
2,2,width,,1,None,0
3,3,dom_x,,1,None,0
4,4,dom_y,,1,None,0
5,5,dom_z,,1,None,0
6,6,pmt_area,,1,None,0
7,7,rde,,1,None,0
8,8,is_bright_dom,,1,None,0
9,9,is_bad_dom,,1,None,0


In [3]:
pd.read_sql("PRAGMA table_info(truth);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,energy,,1,None,0
1,1,position_x,,1,None,0
2,2,position_y,,1,None,0
3,3,position_z,,1,None,0
4,4,azimuth,,1,None,0
5,5,zenith,,1,None,0
6,6,pid,,1,None,0
7,7,event_time,,1,None,0
8,8,sim_type,,1,None,0
9,9,interaction_type,,1,None,0


In [ ]:
from tqdm.notebook import tqdm

# Anzahl der Events in jeder Tabelle zählen (mit Ladebalken)
tables_to_count = {
    "truth": "SELECT COUNT(DISTINCT event_no) AS n_events FROM truth;",
    "SplitInIcePulses": "SELECT COUNT(DISTINCT event_no) AS n_events FROM SplitInIcePulses;",
}

results = {}
for table, query in tqdm(tables_to_count.items(), desc="Zähle Events", unit="table"):
    results[table] = pd.read_sql(query, conn)["n_events"].iloc[0]

for table, n in results.items():
    print(f"Events in {table + ':':<25s} {n:>12,}")

Zähle Events:   0%|          | 0/2 [00:00<?, ?table/s]

In [6]:
# Beispiel: 5 Events laden mit relevanten Größen für Muon Track Reco

# Pulse-Features (Input für das Netzwerk)
pulse_cols = "event_no, dom_x, dom_y, dom_z, dom_time, charge, hlc"

# Truth-Targets (das was rekonstruiert werden soll)
truth_cols = "event_no, azimuth, zenith, energy, position_x, position_y, position_z, track_length"

sample_events = pd.read_sql(
    f"SELECT DISTINCT event_no FROM truth LIMIT 5;", conn
)["event_no"].tolist()

placeholders = ",".join(str(e) for e in sample_events)

pulses = pd.read_sql(
    f"SELECT {pulse_cols} FROM SplitInIcePulses WHERE event_no IN ({placeholders});", conn
)
truth = pd.read_sql(
    f"SELECT {truth_cols} FROM truth WHERE event_no IN ({placeholders});", conn
)

print(f"Pulse-Daten: {len(pulses)} Pulse aus {pulses['event_no'].nunique()} Events")
print("--- Pulses (erste Zeilen) ---")
display(pulses.head(10))

print("\n--- Truth (Targets) ---")
display(truth)

Pulse-Daten: 580 Pulse aus 5 Events
--- Pulses (erste Zeilen) ---


,event_no,dom_x,dom_y,dom_z,dom_time,charge,hlc
0,0,-9.130000,-481.739990,90.739998,6580.0,1.125,0.0
1,0,114.389999,-461.989990,39.099998,10602.0,0.625,0.0
2,0,-290.660004,-307.380005,-74.559998,7716.0,0.525,0.0
3,0,-290.660004,-307.380005,-159.660004,13461.0,0.775,0.0
4,0,-290.660004,-307.380005,-159.660004,16361.0,0.975,0.0
5,0,-43.270000,-267.519989,-214.750000,16432.0,0.875,0.0
6,0,210.470001,-209.770004,411.839996,15847.0,0.975,0.0
7,0,326.850006,-209.070007,345.239990,12531.0,0.325,0.0
8,0,326.850006,-209.070007,-199.429993,11684.0,2.425,1.0
9,0,326.850006,-209.070007,-199.429993,11696.0,0.375,1.0



--- Truth (Targets) ---


,event_no,azimuth,zenith,energy,position_x,position_y,position_z,track_length
0,0,1.156920,1.138356,265.757337,325.410034,-197.784785,-212.029704,832.167698
1,1,1.127122,0.495598,184.850008,-448.207850,-199.543856,140.783221,749.378568
2,2,2.802480,1.205342,2712.835761,1554.454074,-637.145651,-1123.199227,2648.371077
3,3,4.915968,0.518382,382.360145,45.716140,533.279320,-319.592125,1288.928741
4,4,2.473094,0.700460,200.517151,-405.136031,-94.317492,117.652528,731.468244


In [7]:
# Check: Sind die event_no in beiden Tabellen 1:1 gematcht?
events_only_in_truth = pd.read_sql("""
    SELECT COUNT(*) AS n FROM truth
    WHERE event_no NOT IN (SELECT DISTINCT event_no FROM SplitInIcePulses);
""", conn)["n"].iloc[0]

events_only_in_pulses = pd.read_sql("""
    SELECT COUNT(DISTINCT event_no) AS n FROM SplitInIcePulses
    WHERE event_no NOT IN (SELECT event_no FROM truth);
""", conn)["n"].iloc[0]

print(f"Events nur in truth (ohne Pulses):  {events_only_in_truth}")
print(f"Events nur in Pulses (ohne Truth):  {events_only_in_pulses}")

if events_only_in_truth == 0 and events_only_in_pulses == 0:
    print("\nAlles gematcht! Jedes Event hat sowohl Pulse als auch Truth-Info.")
else:
    print("\nACHTUNG: Nicht alle Events sind in beiden Tabellen vorhanden!")

Events nur in truth (ohne Pulses):  0
Events nur in Pulses (ohne Truth):  0

Alles gematcht! Jedes Event hat sowohl Pulse als auch Truth-Info.
